# Go2 Track-Speed Low-Level Training Colab

This notebook is set up for the current oval-track controller workflow. It
pulls your GitHub repo, installs the pinned MuJoCo Playground stack, copies
the Go2 assets, and trains the low-level policy with the track-biased
settings in `configs/colab_runtime_config.json`.

The active curriculum is designed to couple with the high-level CEM limits
you are using: fast straight commands, bounded curve speeds, lateral
commands inside `+-0.25 m/s`, and yaw commands inside `+-0.60 rad/s`.


## 1. Configure Repository URLs

The default repository points at your fork and the `main` branch. Run this
notebook from a fresh Colab runtime; the setup cell will clone the repo into
`/content/EEC289A_Robotics-Homework`.

Colab storage is temporary. The save section near the end copies artifacts
to Google Drive after training.


In [ ]:
from pathlib import Path
import io
import os
import shutil
import subprocess
import sys
import tarfile
import tempfile
import urllib.request
from urllib.parse import urlparse

COURSE_REPO_URL = "https://github.com/yuh-w/EEC289A_Robotics-Homework.git"
COURSE_REPO_BRANCH = "main"
COURSE_REPO_DIR = Path("/content/EEC289A_Robotics-Homework")

PLAYGROUND_REPO = "https://github.com/google-deepmind/mujoco_playground.git"
PLAYGROUND_REF = "dd38c285c6d54266287081e516109f0b15985818"

UNITREE_MUJOCO_REPO = "https://github.com/unitreerobotics/unitree_mujoco.git"
UNITREE_MUJOCO_REF = "1a37b051a10be723405b7ed6dc839361af036d88"

PLAYGROUND_DIR = Path("/content/mujoco_playground")
UNITREE_DIR = Path("/content/unitree_mujoco")

MENAGERIE_REPO = "https://github.com/deepmind/mujoco_menagerie.git"
MENAGERIE_REF = "1b86ece576591213e2b666ebf59508454200ca97"
MENAGERIE_DIR = PLAYGROUND_DIR / "mujoco_playground" / "external_deps" / "mujoco_menagerie"

def run(cmd):
    cmd = [str(part) for part in cmd]
    print("+", " ".join(cmd))
    return subprocess.run(cmd, check=True)

def github_archive_url(repo_url: str, ref: str) -> str:
    repo_path = urlparse(repo_url).path.strip("/")
    if repo_path.endswith(".git"):
        repo_path = repo_path[:-4]
    return f"https://codeload.github.com/{repo_path}/tar.gz/{ref}"

def download_repo_snapshot(repo_url: str, ref: str, target_dir: Path) -> None:
    archive_url = github_archive_url(repo_url, ref)
    print(f"+ download {archive_url}")
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    tmp_dir = Path(tempfile.mkdtemp(prefix=f"{target_dir.name}_", dir=str(target_dir.parent)))
    try:
        with urllib.request.urlopen(archive_url) as response:
            payload = response.read()
        with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
            archive.extractall(tmp_dir)
        extracted_dirs = [path for path in tmp_dir.iterdir() if path.is_dir()]
        if len(extracted_dirs) != 1:
            raise RuntimeError(f"Expected one extracted directory, got {extracted_dirs}")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.move(str(extracted_dirs[0]), str(target_dir))
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

def checkout_existing_repo(target_dir: Path, ref: str) -> None:
    try:
        run(["git", "-C", target_dir, "fetch", "--all", "--tags"])
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git fetch failed for {target_dir}: {exc}. Trying local checkout.")
    run(["git", "-C", target_dir, "checkout", ref])

def ensure_pinned_repo(repo_url: str, ref: str, target_dir: Path) -> None:
    if target_dir.exists() and (target_dir / ".git").exists():
        try:
            checkout_existing_repo(target_dir, ref)
            return
        except subprocess.CalledProcessError as exc:
            print(f"[warn] local git checkout failed for {target_dir}: {exc}. Re-downloading snapshot.")
            shutil.rmtree(target_dir)
    elif target_dir.exists():
        shutil.rmtree(target_dir)

    try:
        run(["git", "clone", repo_url, target_dir])
        checkout_existing_repo(target_dir, ref)
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git path failed for {repo_url}: {exc}. Falling back to archive download.")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, ref, target_dir)

def ensure_course_repo(repo_url: str, branch: str, target_dir: Path) -> None:
    if target_dir.exists():
        return
    try:
        run(["git", "clone", repo_url, target_dir])
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git clone failed for {repo_url}: {exc}. Falling back to archive download.")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, branch, target_dir)

if "google.colab" in sys.modules:
    print("Running inside Colab.")
else:
    print("This notebook was designed for Colab, but local execution may also work.")


## 2. Install system packages and clone repositories

In [ ]:
!command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y ffmpeg)
!python -m pip install -q -U pip setuptools wheel "jedi>=0.16"

import importlib.util
if importlib.util.find_spec("playground") is not None:
    !python -m pip uninstall -y playground

ensure_pinned_repo(PLAYGROUND_REPO, PLAYGROUND_REF, PLAYGROUND_DIR)
ensure_pinned_repo(UNITREE_MUJOCO_REPO, UNITREE_MUJOCO_REF, UNITREE_DIR)
ensure_course_repo(COURSE_REPO_URL, COURSE_REPO_BRANCH, COURSE_REPO_DIR)
ensure_pinned_repo(MENAGERIE_REPO, MENAGERIE_REF, MENAGERIE_DIR)

!python -m pip install -q -r {COURSE_REPO_DIR / "configs" / "colab_requirements.txt"}

%cd {PLAYGROUND_DIR}
!python -m pip install -q -e .

%cd {COURSE_REPO_DIR}
print("Setup finished. Some Colab dependency warnings can usually be ignored if the later checks pass.")


## 3. Copy Go2 assets from `unitree_mujoco` into the local course environment

Use the repo-side helper script so the notebook stays aligned with the code
students will download and edit.


In [ ]:
%cd {COURSE_REPO_DIR}
!python scripts/copy_go2_assets.py --unitree-dir {UNITREE_DIR} --course-dir {COURSE_REPO_DIR}


## 4. Verify The Track-Speed Training Config

This cell prints the active low-level curriculum before training. The
important settings should match the current high-level controller limits:
high forward speed on straights, curve speed around `2.5 m/s`, yaw capped at
`0.60 rad/s`, and lateral commands capped at `0.25 m/s`.


In [ ]:
import json
from pathlib import Path

CONFIG_PATH = COURSE_REPO_DIR / "configs" / "colab_runtime_config.json"
RUN_DIR = COURSE_REPO_DIR / "artifacts" / "run_track_speed"

with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    cfg = json.load(handle)

track_sampler = cfg["stage_2"]["student_stage2_goal"]["track_sampler"]
print("Config:", CONFIG_PATH)
print("Run dir:", RUN_DIR)
print("Stage 1 steps:", cfg.get("runtime_overrides", {}).get("stage_1_num_timesteps", cfg["stage_1"]["num_timesteps"]))
print("Stage 2 steps:", cfg.get("runtime_overrides", {}).get("stage_2_num_timesteps", cfg["stage_2"]["num_timesteps"]))
print("Stage 2 track sampler enabled:", track_sampler["enable"])
print("Straight vx range:", track_sampler["straight_vx"])
print("Curve vx range:", track_sampler["curve_vx"])
print("Curve yaw abs range:", track_sampler["curve_yaw_abs"])
print("Recovery vy range:", track_sampler["recovery_vy"])

%cd {COURSE_REPO_DIR}
!python train.py --config {CONFIG_PATH} --dry-run > /tmp/go2_resolved_config.json
print("Dry-run config resolution succeeded.")


## 5. Run Full Training

This is the main Colab training command. It runs both stages using
`configs/colab_runtime_config.json`: a faster forward warmup followed by the
track-biased stage-2 fine-tuning. The output checkpoint used later is
exported to `artifacts/run_track_speed/best_checkpoint`.


In [ ]:
%cd {COURSE_REPO_DIR}

!python train.py \
  --config {CONFIG_PATH} \
  --stage both \
  --output-dir {RUN_DIR}

CHECKPOINT_DIR = RUN_DIR / "best_checkpoint"
print("Best checkpoint:", CHECKPOINT_DIR)


## 6. Optional: Re-run Stage 2 Only

Use this only after stage 1 has already finished in `RUN_DIR`. The command
restores the selected stage-1 checkpoint automatically, then overwrites the
stage-2 fine-tuning outputs.


In [ ]:
import json
from pathlib import Path

stage1_summary_path = RUN_DIR / "stage_1" / "summary.json"
if not stage1_summary_path.exists():
    raise FileNotFoundError("Run the full training cell first, or point RUN_DIR at a run that contains stage_1/summary.json")

with stage1_summary_path.open("r", encoding="utf-8") as handle:
    stage1_summary = json.load(handle)

RESTORE_CHECKPOINT_DIR = Path(stage1_summary["selected_checkpoint_source"])
print("Restoring stage 2 from:", RESTORE_CHECKPOINT_DIR)

%cd {COURSE_REPO_DIR}

!python train.py \
  --config {CONFIG_PATH} \
  --stage stage_2 \
  --restore-checkpoint-dir {RESTORE_CHECKPOINT_DIR} \
  --output-dir {RUN_DIR}

CHECKPOINT_DIR = RUN_DIR / "best_checkpoint"
print("Best checkpoint:", CHECKPOINT_DIR)


## 7. Restore A Checkpoint And Render A Demo

This cell uses the exported best checkpoint from the training run. The demo
script includes straight, lateral, yaw, and combined command segments so you
can quickly see whether the low-level policy still tracks general joystick
commands after the track-speed fine-tuning.


In [ ]:
from pathlib import Path

CHECKPOINT_DIR = RUN_DIR / "best_checkpoint"
if not CHECKPOINT_DIR.exists():
    raise FileNotFoundError(f"Missing {CHECKPOINT_DIR}. Run training first.")

DEMO_DIR = RUN_DIR / "demo_bundle"

%cd {COURSE_REPO_DIR}

!python test_policy.py \
  --config {CONFIG_PATH} \
  --checkpoint-dir {CHECKPOINT_DIR} \
  --stage-name stage_2 \
  --output-dir {DEMO_DIR}

print("Demo output:", DEMO_DIR)


## 8. Generate The Public Benchmark Rollout

This produces the deterministic public-eval bundle and JSON metrics using
the same checkpoint. The public benchmark is not the same as the oval-track
score, but it is a useful guardrail for tracking error, fall rate, energy,
and foot slip.


In [ ]:
PUBLIC_DIR = RUN_DIR / "public_eval_bundle"

%cd {COURSE_REPO_DIR}

!python generate_public_rollout.py \
  --config {CONFIG_PATH} \
  --checkpoint-dir {CHECKPOINT_DIR} \
  --stage-name stage_2 \
  --output-dir {PUBLIC_DIR} \
  --num-episodes 4 \
  --render-first-episode

!python public_eval.py \
  --config {CONFIG_PATH} \
  --rollout-npz {PUBLIC_DIR / "rollout_public_eval.npz"} \
  --output-json {PUBLIC_DIR / "public_eval.json"}

print("Public eval output:", PUBLIC_DIR)


## 9. Save Artifacts To Google Drive

Run this after training/evaluation. It copies the whole `RUN_DIR` folder to
Drive so the checkpoint, videos, rollout bundle, and metric JSON survive
Colab runtime resets.


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

if "RUN_DIR" not in globals():
    RUN_DIR = COURSE_REPO_DIR / "artifacts" / "run_track_speed"

drive.mount("/content/drive")
DRIVE_SAVE_DIR = Path("/content/drive/MyDrive/go2_track_speed_outputs")
DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

DEST_DIR = DRIVE_SAVE_DIR / RUN_DIR.name
if DEST_DIR.exists():
    shutil.rmtree(DEST_DIR)
shutil.copytree(RUN_DIR, DEST_DIR)
print("Saved artifacts to:", DEST_DIR)


## 10. Optional: Push Notebook/Code Changes From Colab

This is only needed if you edit code inside Colab. Do not hard-code tokens
in the notebook. If you need to push from Colab, store a token as a Colab
secret named `GITHUB_TOKEN`, then run this cell.


In [ ]:
from pathlib import Path
import subprocess

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

if not token:
    print("No GITHUB_TOKEN Colab secret found. Skipping push.")
else:
    %cd {COURSE_REPO_DIR}
    subprocess.run(["git", "config", "user.email", "yuh-w@users.noreply.github.com"], check=True)
    subprocess.run(["git", "config", "user.name", "yuh-w"], check=True)
    subprocess.run(["git", "status", "--short"], check=True)
    subprocess.run([
        "git",
        "add",
        "go2_pg_env/joystick.py",
        "course_common.py",
        "configs/colab_runtime_config.json",
        "configs/course_config.json",
        "notebooks/go2_teaching_colab2.ipynb",
    ], check=True)
    subprocess.run(["git", "commit", "-m", "Update Go2 track-speed Colab workflow"], check=False)
    remote = f"https://{token}@github.com/yuh-w/EEC289A_Robotics-Homework.git"
    subprocess.run(["git", "push", remote, "main"], check=True)
    token = None


## 11. Plot Public-Eval Tracking

This quick plot uses the keys written by `generate_public_rollout.py`:
`command_lin_vel_xy`, `measured_lin_vel_xy`, `command_yaw_rate`, and
`measured_yaw_rate`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rollout_path = PUBLIC_DIR / "rollout_public_eval.npz"
data = np.load(str(rollout_path), allow_pickle=True)

command_xy = data["command_lin_vel_xy"]
measured_xy = data["measured_lin_vel_xy"]
command_yaw = data["command_yaw_rate"]
measured_yaw = data["measured_yaw_rate"]
time = np.arange(len(command_yaw)) * 0.02

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(time, command_xy[:, 0], label="command vx")
axes[0].plot(time, measured_xy[:, 0], label="measured vx")
axes[0].set_ylabel("vx (m/s)")
axes[0].legend()

axes[1].plot(time, command_xy[:, 1], label="command vy")
axes[1].plot(time, measured_xy[:, 1], label="measured vy")
axes[1].set_ylabel("vy (m/s)")
axes[1].legend()

axes[2].plot(time, command_yaw, label="command yaw")
axes[2].plot(time, measured_yaw, label="measured yaw")
axes[2].set_xlabel("time (s)")
axes[2].set_ylabel("yaw rate (rad/s)")
axes[2].legend()
plt.tight_layout()
plt.show()
